# Day 4 — Quantization Benchmark
**Goal:** Compare FP16 → INT8 → INT4 using the same llama.cpp GGUF inference path on this Blackwell box.

| Variant | Method | Expected VRAM |
|---------|--------|---------------|
| FP16 | llama-cpp-python + FP16 GGUF | ~14–15 GB |
| INT8 | llama-cpp-python + Q8_0 GGUF | ~7–8 GB |
| INT4 GGUF | llama-cpp-python Q4_K_M | ~4–5 GB |

**Required artifacts:** FP16 GGUF + Q4_K_M GGUF. If Q8_0 is missing, the benchmark will build it from the FP16 GGUF automatically.**

In [1]:
import sys, os
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent
sys.path.insert(0, str(PROJECT_ROOT / 'src'))
os.chdir(PROJECT_ROOT)
from nyaya_pipeline import config

## Section 1 — Run All Three Benchmarks

Runs FP16, INT8, and INT4 in a single pass through the isolated GGUF worker. This avoids the Blackwell cuBLAS and Unsloth patching failures described in `document.md`.

In [2]:
from nyaya_pipeline.benchmark import run_benchmark, plot_benchmark

# All three variants run via llama.cpp in isolated subprocesses.
# FP16 uses the existing F16 GGUF in /mnt/f, INT8 auto-builds Q8_0 if needed,
# and INT4 resolves through the adapters symlink.
adapter_dir = Path('adapters')
gguf_path   = adapter_dir / 'nyayagpt-q4km.gguf'
fp16_gguf_path = Path('/mnt/f/NyayaGPT-scratch/nyayagpt-fp16.gguf')
int8_gguf_path = Path('/mnt/f/NyayaGPT-scratch/nyayagpt-q8_0.gguf')

results = run_benchmark(
    n_samples=15,
    adapter_dir=adapter_dir,
    fp16_gguf_path=fp16_gguf_path,
    int8_gguf_path=int8_gguf_path,
    gguf_path=gguf_path,
    output_path=config.OUTPUT_DIR / 'benchmark_results.json',
)

2026-04-23 12:39:37,454 [INFO] nyaya_pipeline.benchmark: Running benchmark on 15 eval samples …
2026-04-23 12:39:37,455 [INFO] nyaya_pipeline.benchmark: [FP16] Loading nyayagpt-fp16.gguf (isolated subprocess) …
2026-04-23 12:41:26,160 [INFO] nyaya_pipeline.benchmark: [INT8] Q8_0 GGUF missing; quantizing nyayagpt-fp16.gguf -> nyayagpt-q8_0.gguf via llama-quantize …


llama_print_build_info: build = 1 (84652b8)
llama_print_build_info: built with GNU 9.4.0 for Linux x86_64
main: quantizing '/mnt/f/NyayaGPT-scratch/nyayagpt-fp16.gguf' to '/mnt/f/NyayaGPT-scratch/nyayagpt-q8_0.gguf' as Q8_0
llama_model_loader: loaded meta data with 30 key-value pairs and 291 tensors from /mnt/f/NyayaGPT-scratch/nyayagpt-fp16.gguf (version GGUF V3 (latest))
llama_model_loader: Dumping metadata keys/values. Note: KV overrides do not apply in this output.
llama_model_loader: - kv   0:                       general.architecture str              = llama
llama_model_loader: - kv   1:                               general.type str              = model
llama_model_loader: - kv   2:                               general.name str              = Merged_Model
llama_model_loader: - kv   3:                         general.size_label str              = 7.2B
llama_model_loader: - kv   4:                          llama.block_count u32              = 32
llama_model_loader: - kv   5:    


main: quantize time = 94105.93 ms
main:    total time = 94105.93 ms
2026-04-23 12:43:00,303 [INFO] nyaya_pipeline.benchmark: [INT8] Loading nyayagpt-q8_0.gguf (isolated subprocess) …
2026-04-23 12:43:40,026 [INFO] nyaya_pipeline.benchmark: [INT4 (GGUF)] Loading nyayagpt-q4km.gguf (isolated subprocess) …

Model             Memory (GB)   Latency (ms/tok)    ROUGE-L
--------------------------------------------------------------------
FP16                    14.50               10.8     0.3668
INT8                     7.70                6.8     0.3711
INT4 (GGUF)              4.37                4.9     0.3706
2026-04-23 12:44:04,584 [INFO] nyaya_pipeline.benchmark: Benchmark results saved → /home/ubuntu/Fine-tuning/NyayaGPT/output/benchmark_results.json


In [ ]:
import matplotlib
matplotlib.use('inline')
import matplotlib.pyplot as plt

fig = plot_benchmark(results, save_path=Path('assets/benchmark_chart.png'))
if fig:
    plt.show()
print('Chart saved → assets/benchmark_chart.png')

2026-04-23 12:45:13,235 [INFO] nyaya_pipeline.benchmark: Chart saved → assets/benchmark_chart.png
Chart saved → assets/benchmark_chart.png


## Section 2 — GGUF Conversion (reference)

The baseline GGUF artifacts come from `scripts/build_gguf.py`, which:
1. Loads the winning adapter via Unsloth (auto-merge base + LoRA → FP16)
2. Runs `convert_hf_to_gguf.py` from llama.cpp to produce an FP16 GGUF
3. Quantizes to Q4_K_M via `llama-quantize`

All intermediates (~28 GB: merged FP16 + FP16 GGUF) are written to `/mnt/f/NyayaGPT-scratch/` to avoid growing the WSL VHD. The final 4 GB GGUF lives on `/mnt/f/` and is symlinked into `adapters-2ep/nyayagpt-q4km.gguf`.

The benchmark module reuses that FP16 GGUF directly for the FP16 row and auto-generates `/mnt/f/NyayaGPT-scratch/nyayagpt-q8_0.gguf` for the INT8 row if it is missing.

**Run once (from repo root):**
```bash
python scripts/build_gguf.py
```

**Override adapter / output name:**
```bash
ADAPTER_DIR=adapters-3ep \
  OUTPUT_NAME=nyayagpt-3ep-q4km.gguf \
  python scripts/build_gguf.py
```

**Reclaim ~28 GB after success:**
```bash
rm -rf /mnt/f/NyayaGPT-scratch/merged_model \
       /mnt/f/NyayaGPT-scratch/nyayagpt-fp16.gguf
```

## Section 3 — Memory Profiling

In [4]:
import torch

if torch.cuda.is_available():
    total_gb = torch.cuda.get_device_properties(0).total_memory / 1e9
    alloc_gb = torch.cuda.memory_allocated(0) / 1e9
    reserved_gb = torch.cuda.memory_reserved(0) / 1e9
    
    print(f'GPU Memory Summary:')
    print(f'  Total:    {total_gb:.1f} GB')
    print(f'  Allocated:{alloc_gb:.2f} GB')
    print(f'  Reserved: {reserved_gb:.2f} GB')
    print(f'  Free:     {total_gb - reserved_gb:.1f} GB')

GPU Memory Summary:
  Total:    34.2 GB
  Allocated:0.00 GB
  Reserved: 0.00 GB
  Free:     34.2 GB


## ✅ Day 4 Complete

**Chart:** `assets/benchmark_chart.png` (FP16 + INT8 + INT4)

**Results:** `output/benchmark_results.json`

**Next:** Run `05_ab_test_dashboard.ipynb`